# Small open LLMs on Northeast Indian languages — Colab runner

Runtime → Change runtime type → **T4 GPU**. Each model takes roughly 1–2 h for the full grid; outputs are appended and resumable, so a disconnect loses at most one batch. Results are saved to Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RESULTS = '/content/drive/MyDrive/assamese_research_results'

In [ ]:
!git clone https://github.com/gourav-ai-engineer/assamese_research.git
%cd assamese_research
!pip install -q -e '.[gpu,mt]'

Gemma 3 and Llama 3.2 are gated: accept their licences on the Hugging Face Hub, then log in with a read token.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
!northeast-eval prepare
!ln -sfn $RESULTS results_drive && mkdir -p $RESULTS

## Smoke test (10 items per file) — check outputs look sane before the full run

In [ ]:
!northeast-eval run --model gemma-3-1b-it --limit 10 --out-dir /content/smoke
!head -n 3 /content/smoke/google__gemma-3-1b-it/translation_en-as__zero_shot.jsonl | cut -c1-600

## Full grid — run one model per session if needed

In [ ]:
MODEL = 'gemma-3-1b-it'  # gemma-4-E2B-it, gemma-4-E4B-it, gemma-3-4b-it, Qwen3-1.7B, Qwen3-4B, Llama-3.2-1B, Llama-3.2-3B, sarvam-1
!northeast-eval run --model $MODEL --out-dir $RESULTS/predictions

## Dedicated MT reference line and tokenizer fertility (run once)

In [ ]:
!northeast-eval mt-baseline --backend indictrans2 --out-dir $RESULTS/predictions
!northeast-eval fertility --output $RESULTS/fertility.csv

## Score, tables, figures, annotation sample

In [ ]:
!northeast-eval score --pred-dir $RESULTS/predictions --output $RESULTS/scores.csv
!northeast-eval report --scores $RESULTS/scores.csv --fertility-csv $RESULTS/fertility.csv --out-dir $RESULTS
!northeast-eval sample-errors --pred-dir $RESULTS/predictions --output $RESULTS/annotation/sample.csv
!cat $RESULTS/tables/tables.md